In [19]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as opt
from torch.utils.data import DataLoader,TensorDataset
from torch.nn.utils.rnn import pad_packed_sequence,pack_padded_sequence
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from sklearn import metrics
from sklearn.model_selection import train_test_split
import scipy.io as sio
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [20]:
blosum62 = {
        'A': [4, -1, -2, -2, 0,  -1, -1, 0, -2,  -1, -1, -1, -1, -2, -1, 1,  0,  -3, -2, 0],  # A
        'R': [-1, 5,  0, -2, -3, 1,  0,  -2, 0,  -3, -2, 2,  -1, -3, -2, -1, -1, -3, -2, -3], # R
        'N': [-2, 0,  6,  1,  -3, 0,  0,  0,  1,  -3, -3, 0,  -2, -3, -2, 1,  0,  -4, -2, -3], # N
        'D': [-2, -2, 1,  6,  -3, 0,  2,  -1, -1, -3, -4, -1, -3, -3, -1, 0,  -1, -4, -3, -3], # D
        'C': [0,  -3, -3, -3, 9,  -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1], # C
        'Q': [-1, 1,  0,  0,  -3, 5,  2,  -2, 0,  -3, -2, 1,  0,  -3, -1, 0,  -1, -2, -1, -2], # Q
        'E': [-1, 0,  0,  2,  -4, 2,  5,  -2, 0,  -3, -3, 1,  -2, -3, -1, 0,  -1, -3, -2, -2], # E
        'G': [0,  -2, 0,  -1, -3, -2, -2, 6,  -2, -4, -4, -2, -3, -3, -2, 0,  -2, -2, -3, -3], # G
        'H': [-2, 0,  1,  -1, -3, 0,  0,  -2, 8,  -3, -3, -1, -2, -1, -2, -1, -2, -2, 2,  -3], # H
        'I': [-1, -3, -3, -3, -1, -3, -3, -4, -3, 4,  2,  -3, 1,  0,  -3, -2, -1, -3, -1, 3],  # I
        'L': [-1, -2, -3, -4, -1, -2, -3, -4, -3, 2,  4,  -2, 2,  0,  -3, -2, -1, -2, -1, 1],  # L
        'K': [-1, 2,  0,  -1, -3, 1,  1,  -2, -1, -3, -2, 5,  -1, -3, -1, 0,  -1, -3, -2, -2], # K
        'M': [-1, -1, -2, -3, -1, 0,  -2, -3, -2, 1,  2,  -1, 5,  0,  -2, -1, -1, -1, -1, 1],  # M
        'F': [-2, -3, -3, -3, -2, -3, -3, -3, -1, 0,  0,  -3, 0,  6,  -4, -2, -2, 1,  3,  -1], # F
        'P': [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4, 7,  -1, -1, -4, -3, -2], # P
        'S': [1,  -1, 1,  0,  -1, 0,  0,  0,  -1, -2, -2, 0,  -1, -2, -1, 4,  1,  -3, -2, -2], # S
        'T': [0,  -1, 0,  -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1, 1,  5,  -2, -2, 0],  # T
        'W': [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1, 1,  -4, -3, -2, 11, 2,  -3], # W
        'Y': [-2, -2, -2, -3, -2, -1, -2, -3, 2,  -1, -1, -2, -1, 3,  -3, -2, -2, 2,  7,  -1], # Y
        'V': [0,  -3, -3, -3, -1, -2, -2, -3, -3, 3,  1,  -2, 1,  -1, -2, -2, 0,  -3, -1, 4],  # V
        '-': [0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # -
    }

def normalization(dataset):
    min = dataset.min(axis=0)
    max = dataset.max(axis=0)
    dataset = (dataset - min) / (max - min)
    return dataset

def get_blosum62(seq):
    blosum_list = []
    for i in seq: 
        blosum_list.append(blosum62[i])
    blosum = np.array(blosum_list)
#     blosum = normalization(blosum)
    feature = np.zeros((1002,20))
    idx = blosum.shape[0]
    feature[0:idx,:] = blosum
    return feature


def make_tensor(path):
    data = pd.read_csv(path)
    sequences = data['sequence'].values
    labels = data['label'].values
    evolution = torch.zeros(len(sequences),1002,20)
    lengths = []
    for i in range(len(sequences)):
        lengths.append((len(sequences[i])))
        temp = get_blosum62(sequences[i])
        evolution[i,:,:] = torch.Tensor(temp)

    return evolution,torch.Tensor(lengths),torch.Tensor(labels) 

In [22]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from Bio import SeqIO

# =========================
# 1. FASTA → DataFrame
# =========================
def load_fasta_to_df(path):
    data = []
    for rec in SeqIO.parse(path, "fasta"):
        label = 1 if rec.id.lower().startswith("pos") else 0
        data.append({"sequence": str(rec.seq), "label": label})
    return pd.DataFrame(data)



# =========================
# 2. Amino Acid Encoding
# =========================
AA_DICT = {
    'A':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'K':9,'L':10,
    'M':11,'N':12,'P':13,'Q':14,'R':15,'S':16,'T':17,'V':18,'W':19,'Y':20
}


# =========================
# 3. DataFrame → Tensor
# =========================
import numpy as np

def df_to_tensor_blosum(df, max_len=50):
    X, L = [], []
    for seq in df.sequence:
        mat = [blosum62.get(a, blosum62['-']) for a in seq]
        L.append(len(mat))
        mat = mat[:max_len] + [blosum62['-']]*(max_len-len(mat))
        X.append(mat)
    return (
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(L, dtype=torch.long),
        torch.tensor(df.label.values, dtype=torch.long)
    )


# =========================
# 4. Load FASTA datasets
# =========================
train_df = load_fasta_to_df("data/training_dataset.fasta")
test_df  = load_fasta_to_df("data/independent_dataset.fasta")

train_X, train_L, train_y = df_to_tensor_blosum(train_df)
test_X, test_L, test_y = df_to_tensor_blosum(test_df)

train_FEGS = torch.zeros(len(train_X), 1)
test_FEGS  = torch.zeros(len(test_X), 1)

train_data = DataLoader(
    TensorDataset(train_X, train_L, train_FEGS, train_y),
    batch_size=100, shuffle=True
)

test_data = DataLoader(
    TensorDataset(test_X, test_L, test_FEGS, test_y),
    batch_size=100
)


# =========================
# 5. Dummy FEGS (if not used)
# =========================
train_FEGS = torch.zeros(len(train_pssm), 1)
test_FEGS  = torch.zeros(len(test_pssm), 1)


# =========================
# 6. DataLoader
# =========================
train_data = DataLoader(
    TensorDataset(train_pssm, train_len, train_FEGS, train_label),
    batch_size=100,
    shuffle=True
)

test_data = DataLoader(
    TensorDataset(test_pssm, test_len, test_FEGS, test_label),
    batch_size=100
)

print("data done")


# =========================
# 7. Sanity check
# =========================
print("Train shape:", train_pssm.shape)
print("Train lengths:", train_len[:5])
print("Train labels:", train_label[:5])


data done
Train shape: torch.Size([10084, 1002, 20])
Train lengths: tensor([193.,  62.,  70., 515.,  45.])
Train labels: tensor([0., 1., 1., 0., 1.])


In [23]:
# data = pd.read_csv('data/peptide.csv')
# labels = data['label'].values
# sequences = data['sequence'].values
# fegs = sio.loadmat('data/peptide3864.mat')
# X_train, X_test, y_train1, y_test1 = train_test_split(
#     fegs['FV'], labels, test_size=0.15, random_state=66, stratify=labels)

# train_FEGS = torch.Tensor(X_train)
# test_FEGS = torch.Tensor(X_test)

# S_train, S_test, y_train2, y_test2 = train_test_split(
#     sequences, labels, test_size=0.15, random_state=66, stratify=labels)

# train_pssm, train_len,train_label = make_tensor(S_train,y_train2)
# test_pssm, test_len,test_label = make_tensor(S_test,y_test2)

# train_data = DataLoader(TensorDataset(train_pssm, train_len,train_FEGS,train_label), batch_size=100, shuffle=True)
# test_data = DataLoader(TensorDataset(test_pssm, test_len,test_FEGS, test_label), batch_size=100)

In [30]:
class dvib(nn.Module):
    def __init__(self,k,out_channels, hidden_size):
        super(dvib, self).__init__()
        
        self.conv = torch.nn.Conv2d(in_channels=1,
                            out_channels = out_channels,
                            kernel_size = (1,20),
                            stride=(1,1),
                            padding=(0,0),
                            )
        
        self.rnn = torch.nn.GRU(input_size = out_channels,  
                                hidden_size = hidden_size,
                                num_layers = 2,
                                bidirectional = True,
                                batch_first = True,
                                dropout = 0.2
                              )
        
        self.fc1 = nn.Linear(hidden_size*4, hidden_size*4)
#         self.fc2 = nn.Linear(1024,1024)
        self.enc_mean = nn.Linear(hidden_size*4+1,k)
        self.enc_std = nn.Linear(hidden_size*4+1,k)
        self.dec = nn.Linear(k, 2)
        
        self.drop_layer = torch.nn.Dropout(0.5)
        
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.constant_(self.fc1.bias, 0.0)
#         nn.init.xavier_uniform_(self.fc2.weight)
#         nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_mean.weight)
        nn.init.constant_(self.enc_mean.bias, 0.0)
        nn.init.xavier_uniform_(self.enc_std.weight)
        nn.init.constant_(self.enc_std.bias, 0.0)
        nn.init.xavier_uniform_(self.dec.weight)
        nn.init.constant_(self.dec.bias, 0.0)
   
        
        
    def cnn_gru(self,x,lens):
        x = x.unsqueeze(1)
#         print(x.shape)
        x = self.conv(x)
#         print(x.shape)   
        x = torch.nn.ReLU()(x)
#         print(x.shape,type(x))
        x = x.squeeze(3)
#         x = x.view(x.size(0),-1)
        x = x.permute(0,2,1)
#         print(x.shape)
#         print(type(lens))
        gru_input = pack_padded_sequence(x,lens,batch_first=True)
        output, hidden = self.rnn(gru_input)
#         print(hidden.shape)
        output_all = torch.cat([hidden[-1],hidden[-2],hidden[-3],hidden[-4]],dim=1)
#         print("output_all.shape:",output_all.shape)    
        return output_all
    
        
    def forward(self, pssm, lengths, FEGS): 
        cnn_vectors = self.cnn_gru(pssm,lengths)
        feature_vec = torch.cat([cnn_vectors,FEGS],dim = 1)
      
        enc_mean, enc_std = self.enc_mean(feature_vec), f.softplus(self.enc_std(feature_vec)-5)
        eps = torch.randn_like(enc_std)
        latent = enc_mean + enc_std*eps
        outputs = self.dec(latent)   # sigmoid رو حذف کن
        return outputs, enc_mean, enc_std, latent


In [28]:
CE = nn.CrossEntropyLoss(reduction='sum')
betas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6,1e-7]

def calc_loss(y_pred, labels, enc_mean, enc_std, beta=1e-3):
    """    
    y_pred : [batch_size,2]
    label : [batch_size,1]    
    enc_mean : [batch_size,z_dim]  
    enc_std: [batch_size,z_dim] 
    """   
    
    ce = CE(y_pred,labels)
    KL = 0.5 * torch.sum(enc_mean.pow(2) + enc_std.pow(2) - 2*enc_std.log() - 1)
    
    return (ce + beta * KL)/y_pred.shape[0]

In [ ]:
out_channels = 128
hidden_size = 512
beta = 2  # 0,1,3,4,5,6
k = 1024
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = dvib(k=1024, out_channels=128, hidden_size=512).to(device)
optimizer = opt.Adam(model.parameters(), lr=1e-4)

for epoch in range(50):
    model.train()
    total_loss, correct = 0, 0

    for seqs, lens, fegs, labels in train_data:
        lens, idx = lens.sort(descending=True)
        seqs = seqs[idx].to(device)
        fegs = fegs[idx].to(device)
        labels = labels[idx].to(device)
        labels = labels.long()
        lens = lens.to(device)

        y_pred, mu, std, _ = model(seqs, lens, fegs)
        loss = calc_loss(y_pred, labels, mu, std, beta=1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (y_pred.argmax(1) == labels).sum().item()

    print(f"Epoch {epoch} | Loss {total_loss:.4f} | Acc {correct/len(train_data.dataset):.4f}")

    model.eval()
    y_true, y_hat = [], []
    with torch.no_grad():
        for seqs, lens, fegs, labels in test_data:
            lens, idx = lens.sort(descending=True)
            seqs = seqs[idx].to(device)
            fegs = fegs[idx].to(device)
            labels = labels[idx].to(device)
            lens = lens.to(device)

            y_pred, *_ = model(seqs, lens, fegs)
            y_hat += y_pred.argmax(1).cpu().tolist()
            y_true += labels.cpu().tolist()

    print(
        "Test | Acc:",
        metrics.accuracy_score(y_true, y_hat),
        "F1:",
        metrics.f1_score(y_true, y_hat),
        "MCC:",
        metrics.matthews_corrcoef(y_true, y_hat)
    )


Epoch 0 | Loss 86774.3256 | Acc 0.5370
Test | Acc: 0.551440329218107 F1: 0.13262599469496023 MCC: -0.007470959938466959
